In [0]:
# ============================================================
# NOTEBOOK   : silver_customers
# PURPOSE    : Clean and transform bronze.customers table
# SOURCE     : fincompliance_catalog.bronze.customers
# DESTINATION: fincompliance_catalog.silver.customers
# TRANSFORMATIONS:
#   - Deduplicate on customer_id
#   - Trim and lowercase city names
#   - Uppercase state codes
#   - Handle null values
#   - Add customer_state_name (full name)
#   - Add customer_region (geographic region)
#   - Drop bronze audit columns
#   - Add silver_updated_at audit column
# ============================================================

In [0]:

%run ../configs/config

In [0]:
## AUTHENTICATE AND SET CATALOG
# Authenticate Spark to ADLS Gen2 (asadev)
# Set active catalog to fincompliance_catalog

authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,              # refers to a column by name
    trim,             # removes leading and trailing spaces
    lower,            # converts text to lowercase
    upper,            # converts text to uppercase
    when,             # used for if/else conditions
    lit,              # converts python value to spark column
    current_timestamp,# gets current date and time
    count,            # counts rows
    sum as spark_sum  # adds up values
)

In [0]:
## Reading Bronze Customers Table
df_customers = spark.table(f"{BRONZE_DB}.customers")

print("="*60)
print("BRONZE.CUSTOMERS - Before Transformation")
print("="*60)
print(f"Total Rows : {df_customers.count()}")
print(f"Total columns : {len(df_customers.columns)}")
print(f"\n Column names :")

for col_name in df_customers.columns:
    print(f"  → {col_name}")

print("\nSample data BEFORE transformation (3 rows):")
df_customers.show(3, truncate=True)

In [0]:
# First Transformation = DEDUPLICATION
# Remove duplicate rows based on customer_id
# customer_id is the primary key — must be unique

rows_before = df_customers.count()
print(f"Rows BEFORE deduplication : {rows_before:,}")

df_customers_silver = df_customers.dropDuplicates(["customer_id"])

# Count rows after deduplication
rows_after = df_customers_silver.count()
print(f"Rows AFTER deduplication  : {rows_after:,}")

## Show how many duplicates were removed
duplicates_removed = rows_before - rows_after
print(f"Duplicates removed : {duplicates_removed:,}")

In [0]:
# CELL 7 - TRIM CITY NAMES
# Remove leading and trailing spaces from city names
# "  sao paulo  " → "sao paulo"

# Show city names BEFORE trim
print("City names BEFORE trim (5 rows):")
df_customers_silver.select("customer_city").show(5, truncate=False)

# Apply trim transformation
df_customers_silver = df_customers_silver.withColumn(
    "customer_city",
    trim(col("customer_city"))
)

# Show city names AFTER trim
print("City names AFTER trim (5 rows):")
df_customers_silver.select("customer_city").show(5, truncate=False)

print("Trim transformation complete")

In [0]:
# LOWERCASE CITY NAMES
# Standardise all city names to lowercase

# Show city names before lowercase
print("City names BEFORE lowercase (5 rows):")
df_customers_silver.select("customer_city").show(5, truncate=False)

# Apply lowercase transformation
df_customers_silver = df_customers_silver.withColumn(
    "customer_city",
    lower(col("customer_city"))
)

# Show city names AFTER lowercase
print("City names AFTER lowercase (5 rows):")
df_customers_silver.select("customer_city").show(5, truncate=False)

# Show how many unique cities we have
unique_cities = df_customers_silver \
    .select("customer_city") \
    .distinct() \
    .count()
print(f"\nTotal unique cities found : {unique_cities:,}")
print("Lowercase transformation complete")

In [0]:
# UPPERCASE STATE CODES
# Standardise all state codes to uppercase
# Also trim spaces from state codes

# Show state codes BEFORE uppercase
print("State codes BEFORE uppercase (5 rows):")
df_customers_silver.select("customer_state").show(5, truncate=False)

# Show unique state codes before
print("Unique states BEFORE uppercase:")
df_customers_silver \
    .select("customer_state") \
    .distinct() \
    .orderBy("customer_state") \
    .show(30, truncate=False)

# Apply trim and uppercase transformation
df_customers_silver = df_customers_silver.withColumn(
    "customer_state",
    upper(trim(col("customer_state"))))

# Show state codes AFTER uppercase
print("State codes AFTER uppercase (5 rows):")
df_customers_silver.select("customer_state").show(5, truncate=False)

# Show unique state codes after
print("Unique states AFTER uppercase:")
df_customers_silver \
    .select("customer_state") \
    .distinct() \
    .orderBy("customer_state") \
    .show(30, truncate=False)

# Count unique states
unique_states = df_customers_silver \
    .select("customer_state") \
    .distinct() \
    .count()
print(f"Total unique states found : {unique_states:,}")
print("Uppercase transformation complete")




In [0]:
# HANDLE NULL CITY NAMES
# Check for null city names and replace with "unknown"
# Never leave nulls in dimension columns

# Count nulls BEFORE handling
nulls_before = df_customers_silver \
    .filter(col("customer_city").isNull()) \
    .count()
print(f"Null city names BEFORE : {nulls_before:,}")

# Apply null handling transformation
df_customers_silver = df_customers_silver.withColumn(
    "customer_city",
    when(col("customer_city").isNull(), lit("unknown"))
    .otherwise(col("customer_city"))
)

# Count nulls AFTER handling
nulls_after = df_customers_silver \
    .filter(col("customer_city").isNull()) \
    .count()
print(f"Null city names AFTER  : {nulls_after:,}")

print(f"Nulls handled: {nulls_before - nulls_after:,}")
print("Null city names handled")


In [0]:
# TRANSFORMATION TO HANDLE NULL STATE CODES
# Check for null state codes and replace with "unknown"

# Count nulls BEFORE handling
nulls_before = df_customers_silver \
    .filter(col("customer_state").isNull()) \
    .count()
print(f"Null state codes BEFORE : {nulls_before:,}")

# Apply null handling transformation
df_customers_silver = df_customers_silver.withColumn(
    "customer_state",
    when(
        col("customer_state").isNull() |
        (col("customer_state") == ""),
        lit("unknown")
    )
    .otherwise(col("customer_state"))
)

# Count nulls AFTER handling
nulls_after = df_customers_silver \
    .filter(col("customer_state").isNull()) \
    .count()
print(f"Null state codes AFTER  : {nulls_after:,}")
print(f"Nulls handled           : {nulls_before - nulls_after:,}")
print("Null state codes handled")

In [0]:
# VALIDATE ZIP CODE PREFIX
# Brazilian zip codes must be between 1000 and 99999
# Any value outside this range is invalid → set to null

# Count invalid zip codes BEFORE validation
invalid_before = df_customers_silver \
    .filter(
        (col("customer_zip_code_prefix") < 1000) |
        (col("customer_zip_code_prefix") > 99999)
    ) \
    .count()
print(f"Invalid zip codes BEFORE : {invalid_before:,}")

# Show sample of invalid zip codes if any exist
if invalid_before > 0:
    print("\nSample invalid zip codes:")
    df_customers_silver \
        .filter(
            (col("customer_zip_code_prefix") < 1000) |
            (col("customer_zip_code_prefix") > 99999)
        ) \
        .select("customer_id", "customer_zip_code_prefix",
                "customer_city", "customer_state") \
        .show(5, truncate=False)

# Apply zip code validation
df_customers_silver = df_customers_silver.withColumn(
    "customer_zip_code_prefix",
    when(
        (col("customer_zip_code_prefix") < 1000) |
        (col("customer_zip_code_prefix") > 99999),
        None
    )
    .otherwise(col("customer_zip_code_prefix"))
)

# Count invalid zip codes AFTER validation
invalid_after = df_customers_silver \
    .filter(
        (col("customer_zip_code_prefix") < 1000) |
        (col("customer_zip_code_prefix") > 99999)
    ) \
    .count()

print(f"Invalid zip codes AFTER  : {invalid_after:,}")
print(f"Invalid zip codes fixed  : {invalid_before - invalid_after:,}")

# Show zip code statistics
print("\nZip code statistics:")
df_customers_silver \
    .select("customer_zip_code_prefix") \
    .summary("min", "max", "count") \
    .show()

print("Zip code validation complete")

In [0]:
# TRANSFORMATION TO ADD STATE FULL NAME
# Map 2 letter state codes to full state names
# SP → Sao Paulo
# RJ → Rio de Janeiro
# Brazil has 26 states + 1 federal district = 27 total

# Show state codes BEFORE adding full name
print("BEFORE adding state full name (5 rows):")
df_customers_silver \
    .select("customer_state") \
    .show(5, truncate=False)

# Apply state full name transformation
df_customers_silver = df_customers_silver.withColumn(
    "customer_state_name",
    when(col("customer_state") == "SP", "Sao Paulo")
    .when(col("customer_state") == "RJ", "Rio de Janeiro")
    .when(col("customer_state") == "MG", "Minas Gerais")
    .when(col("customer_state") == "RS", "Rio Grande do Sul")
    .when(col("customer_state") == "PR", "Parana")
    .when(col("customer_state") == "SC", "Santa Catarina")
    .when(col("customer_state") == "BA", "Bahia")
    .when(col("customer_state") == "GO", "Goias")
    .when(col("customer_state") == "ES", "Espirito Santo")
    .when(col("customer_state") == "PE", "Pernambuco")
    .when(col("customer_state") == "CE", "Ceara")
    .when(col("customer_state") == "MA", "Maranhao")
    .when(col("customer_state") == "MT", "Mato Grosso")
    .when(col("customer_state") == "MS", "Mato Grosso do Sul")
    .when(col("customer_state") == "PA", "Para")
    .when(col("customer_state") == "RN", "Rio Grande do Norte")
    .when(col("customer_state") == "PB", "Paraiba")
    .when(col("customer_state") == "SE", "Sergipe")
    .when(col("customer_state") == "AL", "Alagoas")
    .when(col("customer_state") == "PI", "Piaui")
    .when(col("customer_state") == "AM", "Amazonas")
    .when(col("customer_state") == "AC", "Acre")
    .when(col("customer_state") == "RO", "Rondonia")
    .when(col("customer_state") == "RR", "Roraima")
    .when(col("customer_state") == "AP", "Amapa")
    .when(col("customer_state") == "TO", "Tocantins")
    .when(col("customer_state") == "DF", "Distrito Federal")
    .otherwise("Unknown")
)

# Show AFTER adding full name
print("\nAFTER adding state full name (5 rows):")
df_customers_silver \
    .select("customer_state", "customer_state_name") \
    .show(5, truncate=False)

# Verify all 27 states are mapped correctly
print("\nAll state mappings:")
df_customers_silver \
    .select("customer_state", "customer_state_name") \
    .distinct() \
    .orderBy("customer_state") \
    .show(30, truncate=False)

# Check if any state mapped to "Unknown"
unknown_states = df_customers_silver \
    .filter(col("customer_state_name") == "Unknown") \
    .count()
print(f"States mapped to Unknown : {unknown_states:,}")
print("State full name transformation complete")

In [0]:
# CELL 14 - ADD CUSTOMER REGION
# Group Brazilian states into 5 geographic regions
# Southeast, South, Northeast, North, Central-West
# Apply customer region transformation
df_customers_silver = df_customers_silver.withColumn(
    "customer_region",
    when(col("customer_state").isin(
        ["SP", "RJ", "MG", "ES"]),
        "Southeast")
    .when(col("customer_state").isin(
        ["RS", "SC", "PR"]),
        "South")
    .when(col("customer_state").isin(
        ["BA", "PE", "CE", "MA", "PI",
         "RN", "PB", "SE", "AL"]),
        "Northeast")
    .when(col("customer_state").isin(
        ["AM", "PA", "AC", "RO",
         "RR", "AP", "TO"]),
        "North")
    .when(col("customer_state").isin(
        ["GO", "MT", "MS", "DF"]),
        "Central-West")
    .otherwise("Unknown")
)

# Show AFTER adding region
print("\nAFTER adding customer region (5 rows):")
df_customers_silver \
    .select(
        "customer_state",
        "customer_state_name",
        "customer_region"
    ) \
    .show(5, truncate=False)

# Show region breakdown — how many customers per region
print("\nCustomer count by region:")
df_customers_silver \
    .groupBy("customer_region") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# Show all state to region mappings
print("\nAll state to region mappings:")
df_customers_silver \
    .select(
        "customer_state",
        "customer_state_name",
        "customer_region"
    ) \
    .distinct() \
    .orderBy("customer_region", "customer_state") \
    .show(30, truncate=False)

# Check for any unknown regions
unknown_regions = df_customers_silver \
    .filter(col("customer_region") == "Unknown") \
    .count()
print(f"States mapped to Unknown : {unknown_regions:,}")
print("Customer region transformation complete")

In [0]:
# CELL 15 - DROP BRONZE AUDIT COLUMNS
# Remove ingestion_timestamp and source_file columns
# These were useful in bronze to track raw ingestion
# They are not needed in silver layer

# Show columns BEFORE dropping
print("Columns BEFORE dropping audit columns:")
for col_name in df_customers_silver.columns:
    print(f"  → {col_name}")

print(f"\nTotal columns BEFORE : {len(df_customers_silver.columns)}")

# Drop bronze audit columns
df_customers_silver = df_customers_silver.drop(
    "ingestion_timestamp",
    "source_file"
)

# Show columns AFTER dropping
print("\nColumns AFTER dropping audit columns:")
for col_name in df_customers_silver.columns:
    print(f"  → {col_name}")

print(f"\nTotal columns AFTER : {len(df_customers_silver.columns)}")
print("Bronze audit columns dropped")

In [0]:
# ADD SILVER AUDIT COLUMN
# Add silver_updated_at to track when this
# silver transformation ran
# Standard practice in every production pipeline

# Show columns BEFORE adding audit column
print("Columns BEFORE adding silver audit column:")
for col_name in df_customers_silver.columns:
    print(f"  → {col_name}")

# Add silver audit column
df_customers_silver = df_customers_silver.withColumn(
    "silver_updated_at",
    current_timestamp()
)
# Show columns AFTER adding audit column
print("\nColumns AFTER adding silver audit column:")
for col_name in df_customers_silver.columns:
    print(f"  → {col_name}")

# Show sample of the audit column
print("\nSample of silver_updated_at (3 rows):")
df_customers_silver \
    .select(
        "customer_id",
        "customer_city",
        "customer_state",
        "customer_region",
        "silver_updated_at"
    ) \
    .show(3, truncate=False)

print(f"Total columns now : {len(df_customers_silver.columns)}")
print(f"Total rows now    : {df_customers_silver.count():,}")
print("Silver audit column added")

In [0]:
# WRITE TO SILVER
# Write the transformed DataFrame to
# fincompliance_catalog.silver.customers
# as a Delta table

print("Writing to silver.customers...")
print(f"Destination : {SILVER_DB}.customers")
print(f"Format      : Delta")
print(f"Mode        : Overwrite")

# Write transformed DataFrame to silver layer
(
    df_customers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.customers")
)

print("\n Successfully written to silver.customers")
print(f"Total rows written : {df_customers_silver.count():,}")
print(f"Total columns      : {len(df_customers_silver.columns)}")

# Show final column list
print("\nFinal columns in silver.customers:")
for col_name in df_customers_silver.columns:
    print(f"  → {col_name}")

In [0]:
# SILVER CUSTOMERS VERIFICATION
# Verify the silver.customers table was written correctly
# Compare bronze vs silver to confirm transformations worked

print("=" * 60)
print("SILVER CUSTOMERS VERIFICATION")
print("=" * 60)

bronze_count = spark.table(f"{BRONZE_DB}.customers").count()
silver_count = spark.table(f"{SILVER_DB}.customers").count()

print(f"\nCHECK 1 — Row counts:")
print(f"  Bronze rows : {bronze_count:,}")
print(f"  Silver rows : {silver_count:,}")

if silver_count == bronze_count:
    print(f"Row counts match")
else:
    print(f"Row counts differ by : {bronze_count - silver_count:,}")

bronze_cols = spark.table(f"{BRONZE_DB}.customers").columns
silver_cols = spark.table(f"{SILVER_DB}.customers").columns

print(f"\nCHECK 2 — Column comparison:")
print(f"  Bronze columns ({len(bronze_cols)}) : {bronze_cols}")
print(f"  Silver columns ({len(silver_cols)}) : {silver_cols}")

# Columns removed from bronze
removed_cols = [c for c in bronze_cols if c not in silver_cols]
print(f"\n  Columns removed : {removed_cols}")

# Columns added in silver
added_cols = [c for c in silver_cols if c not in bronze_cols]
print(f"  Columns added   : {added_cols}")

print(f"\nCHECK 3 — Null checks:")
df_silver = spark.table(f"{SILVER_DB}.customers")

key_columns = [
    "customer_id",
    "customer_city",
    "customer_state",
    "customer_state_name",
    "customer_region"
]

for column in key_columns:
    null_count = df_silver \
        .filter(col(column).isNull()) \
        .count()
    if null_count == 0:
        print(f"{column:<30} : no nulls")
    else:
        print(f"{column:<30} : {null_count:,} nulls found")

print(f"\nCHECK 4 — Region distribution:")
df_silver \
    .groupBy("customer_region") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

print(f"CHECK 5 — State name sample:")
df_silver \
    .select(
        "customer_state",
        "customer_state_name",
        "customer_region"
    ) \
    .distinct() \
    .orderBy("customer_region", "customer_state") \
    .show(30, truncate=False)


print(f"\nCHECK 6 — Sample data from silver.customers (3 rows):")
df_silver.show(3, truncate=True)

print("=" * 60)
print("silver.customers verification complete!")
print(f"Table : {SILVER_DB}.customers")
print(f"Rows  : {silver_count:,}")
print(f"Cols  : {len(silver_cols)}")
print("=" * 60)